# Exercise: Classify Sea Animals with Transfer Learning

[Caltech-101](https://data.caltech.edu/records/mzrjq-6wc02) is a collection of about 9,000 photos of objects in 101 categories, collected at the California Institute of Technology in 2003. In this exercise you use five of its categories, all animals that live in water: **crabs, crayfish, dolphins, lobsters and starfish** (all from the sea except the crayfish, which lives in fresh water). Each category has only 41 to 86 photos, far too few to train a CNN from scratch, which makes it a typical case for transfer learning. Three of the five are crustaceans with claws and segmented bodies, so telling them apart is the hard part.

You apply the workflow of notebooks 01 and 02: test a pretrained network as it is, extract features and train a dense head on them, and train a frozen base with augmentation. You compare everything on the validation set and evaluate your chosen model once on the test set. The tasks describe what to build and what to report; the code is yours to write.

## Learning Objectives

By the end of this exercise, you should be able to:

- Load a small image dataset from folders and inspect its classes.
- Train a dense classification head on the features of a frozen pretrained base.
- Train a frozen base with data augmentation.
- Compare the models on the validation set.
- Evaluate the chosen model once on the test set and identify the classes it confuses.

## Setup

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras.applications import mobilenet_v2
from sklearn.metrics import ConfusionMatrixDisplay

keras.utils.set_random_seed(42)
IMAGE_SIZE = (160, 160)

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## Step 1: The Data

The repository contains the photos in `data/sea_animals/`, already split per category into 60 % training, 20 % validation and 20 % test photos, in the folder layout `image_dataset_from_directory` expects: `data/sea_animals/<split>/<class>/`. The cell below counts them.

In [ ]:
CLASSES = ["crab", "crayfish", "dolphin", "lobster", "starfish"]
sea_animals = Path("../data") / "sea_animals"

pd.DataFrame(
    {split: {name: len(list((sea_animals / split / name).glob("*.jpg"))) for name in CLASSES}
     for split in ["train", "validation", "test"]}
)

## Step 2: Load the Images

- Load the three splits with `image_dataset_from_directory` from `data/sea_animals/<split>`, with `IMAGE_SIZE`, a batch size of 32 and integer labels. Shuffle the training set (with `seed=42`), not the validation and test sets.
- Plot one example image of each class with its class name.

Caltech-101 contains a few greyscale photos; `image_dataset_from_directory` converts them to three colour channels by default.

In [ ]:
def load_images(split, shuffle):
    """Return a dataset of (images, labels) batches for one split of the sea animals."""
    return keras.utils.image_dataset_from_directory(
        sea_animals / split, image_size=IMAGE_SIZE, batch_size=32, label_mode="int", shuffle=shuffle, seed=42
    )


train_ds = load_images("train", shuffle=True)
valid_ds = load_images("validation", shuffle=False)
test_ds = load_images("test", shuffle=False)
class_names = train_ds.class_names

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, name in zip(axes, class_names):
    image = keras.utils.load_img(next((sea_animals / "train" / name).glob("*.jpg")))
    ax.imshow(image)
    ax.set_title(name)
    ax.axis("off")
plt.show()

## Step 3: A Pretrained Network as It Is

Classify the first validation photo of each class (in sorted file order) with the complete MobileNetV2 (`weights="imagenet"`, images of 224 × 224 pixels, `preprocess_input` and `decode_predictions` from `mobilenet_v2`), and print its top-3 ImageNet classes.

Which animals does MobileNetV2 recognise? ImageNet has classes for crabs, crayfish, lobsters and starfish, but not for dolphins: what does the network answer for the animals it does not recognise?

In [ ]:
imagenet_model = mobilenet_v2.MobileNetV2(weights="imagenet")
for name in class_names:
    path = sorted((sea_animals / "validation" / name).glob("*.jpg"))[0]
    pixels = keras.utils.img_to_array(keras.utils.load_img(path, target_size=(224, 224)))
    probabilities = imagenet_model.predict(mobilenet_v2.preprocess_input(pixels[np.newaxis]), verbose=0)
    top3 = mobilenet_v2.decode_predictions(probabilities, top=3)[0]
    print(f"{name:10s}", ", ".join(f"{label} {p:.2f}" for _, label, p in top3))

**Result.** Only the lobster is recognised with confidence (American lobster, 0.85). ImageNet has classes for crabs, crayfish and starfish as well, but for these photos MobileNetV2 is unsure or wrong: the crab becomes a tick (0.47) with rock crab second, the crayfish an American lobster (0.29) with crayfish only third (0.07), and the starfish a feather boa (0.20). ImageNet has no dolphin class at all, so the dolphin becomes a great white shark (0.29), the closest class it has. The features of the network are useful, but its 1,000 classes do not match this task.

## Step 4: Features and a Dense Head

- Create the MobileNetV2 base without its classifier, with `pooling="avg"` and `IMAGE_SIZE`, and freeze it.
- Extract the feature vectors and labels of the training, validation and test sets. Remember to use an **unshuffled** copy of the training set.
- Train the dense head of notebook 02 on the training features: `Dropout(0.2)` and a softmax output layer for the five classes, with Adam (learning rate 0.001) for 30 epochs. Set the random seed to 42 first.

Create a dictionary `results` and store the validation accuracy of every model in it, starting with this one.

How does the validation accuracy compare with the dense head on the bean leaf features in notebook 02 (0.910), which had about eight times as many training images per class?

In [ ]:
base = mobilenet_v2.MobileNetV2(include_top=False, weights="imagenet", input_shape=IMAGE_SIZE + (3,), pooling="avg")
base.trainable = False


def extract_features(dataset):
    """Run the frozen base over every batch of a dataset; return the feature vectors and the labels."""
    all_features = []
    all_labels = []
    for images, labels in dataset:  # one batch of up to 32 images at a time
        images = mobilenet_v2.preprocess_input(images)  # scale the pixels from 0 to 255 to -1 to 1
        features = base.predict(images, verbose=0)  # one row of 1,280 numbers per image
        all_features.append(features)
        all_labels.append(labels.numpy())
    return np.concatenate(all_features), np.concatenate(all_labels)  # stack the batches into one table


X_train, y_train = extract_features(load_images("train", shuffle=False))
X_valid, y_valid = extract_features(valid_ds)
X_test, y_test = extract_features(test_ds)

keras.utils.set_random_seed(42)
head = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],)),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(units=len(class_names), activation="softmax"),
    ]
)
head.compile(loss="sparse_categorical_crossentropy", optimizer=keras.optimizers.Adam(1e-3), metrics=["accuracy"])
head.fit(X_train, y_train, epochs=30, verbose=0)

results = {}
results["Features + dense head"] = head.evaluate(X_valid, y_valid, verbose=0)[1]
print(f"Validation accuracy: {results['Features + dense head']:.3f}")

**Result.** The dense head on the features reaches a validation accuracy of 0.925, 62 of the 67 validation photos, with only 199 training photos. That is slightly above the dense head on the bean leaf features in notebook 02 (0.910), although the bean data had about eight times as many training images per class: the pretrained features carry most of the knowledge that the small dataset lacks.

## Step 5: Frozen Base with Data Augmentation

Build a `keras.Sequential` model: an augmentation block (`RandomFlip("horizontal")`, `RandomRotation(0.1)`, `RandomZoom(0.1)`), the MobileNetV2 scaling as a `Rescaling(scale=1 / 127.5, offset=-1)` layer, the frozen base, `Dropout(0.2)` and a softmax output layer for the five classes.

Compile it with Adam (learning rate 0.001) and the sparse categorical cross-entropy, train it for 10 epochs on the image datasets, plot the learning curves and store its validation accuracy.

Is a horizontal flip a sensible augmentation for these photos? Would a vertical flip be?

In [ ]:
def plot_curves(logs, title):
    """Plot the training and validation loss and accuracy of one run."""
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
    logs[["loss", "val_loss"]].plot(ax=ax_loss, xlabel="Epoch", title=f"{title}: loss")
    logs[["accuracy", "val_accuracy"]].plot(ax=ax_acc, xlabel="Epoch", title=f"{title}: accuracy")
    plt.show()


keras.utils.clear_session()
keras.utils.set_random_seed(42)
augmentation = keras.Sequential(
    [keras.layers.RandomFlip("horizontal"), keras.layers.RandomRotation(0.1), keras.layers.RandomZoom(0.1)]
)
transfer_model = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (3,)),
        augmentation,
        keras.layers.Rescaling(scale=1 / 127.5, offset=-1),
        base,
        keras.layers.Dropout(0.2),
        keras.layers.Dense(units=len(class_names), activation="softmax"),
    ]
)

transfer_model.compile(loss="sparse_categorical_crossentropy", optimizer=keras.optimizers.Adam(1e-3), metrics=["accuracy"])
history = transfer_model.fit(train_ds, validation_data=valid_ds, epochs=10, verbose=0)
frozen_logs = pd.DataFrame(history.history, index=history.epoch)
plot_curves(frozen_logs, "Frozen base with augmentation")
results["Frozen base + augmentation"] = transfer_model.evaluate(valid_ds, verbose=0)[1]
print(f"Validation accuracy: {results['Frozen base + augmentation']:.3f}")

**Result.** The frozen base with augmentation reaches 0.925. A horizontal flip is sensible: an animal facing left is the same animal facing right. A vertical flip would be sensible for starfish photographed from above, but not for dolphins or crabs, which are almost always photographed the right way up, so it would create images that do not occur in the data.

## Step 6: Compare and Choose

Show the validation accuracies of both models in one table, sorted from best to worst, and choose your final model. The validation set has only 67 images: how many images does a difference of 0.015 correspond to? Take this into account in your choice.

In [ ]:
pd.Series(results, name="Validation accuracy").sort_values(ascending=False).round(3).to_frame()

**Result.** With 67 validation images, one image is 0.015. The dense head on the features and the frozen base with augmentation both reach 0.925, 62 of the 67 photos, so they tie. The dense head is the better choice: it is the simpler model, and it trains in seconds instead of minutes.

## Step 7: Evaluate Once on the Test Set

Evaluate your chosen model on the test set and plot its confusion matrix with the class names. Which classes does it confuse, and why could that be?

In [ ]:
# the chosen model: the dense head on the features (see step 6)
print(f"Dense head on the features, test accuracy: {head.evaluate(X_test, y_test, verbose=0)[1]:.3f}")
y_pred = head.predict(X_test, verbose=0).argmax(axis=1)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=class_names, colorbar=False, ax=ax)
ax.grid(False)
plt.show()

**Result.** The dense head reaches a test accuracy of 0.957, 66 of 69 photos, above its validation accuracy of 0.925. With so few photos, one image more or less changes the accuracy by about 0.015, so this difference is chance and not a sign that the model got better. Crabs, dolphins and starfish are all correct. The three mistakes are all between the crustaceans: two crayfish are predicted as lobsters and one lobster as a crab. Lobsters and crayfish have the same body plan, and the lobster class has the fewest training photos (24), so the model has few examples to learn the differences in size and claw shape.

## Reflection

Answer these questions in the Markdown cell below.

1. The dense head on the features of a frozen base has only one trainable layer. Why can it perform as well as the model with augmentation on this data?
2. With data augmentation, the features cannot be computed once in advance. Why not, and what do you gain and lose with each approach?
3. Your training set has fewer than 60 images per class. What would you expect from a CNN trained from scratch on it, and why?

**Answers.**

1. The features of MobileNetV2 already encode shapes, textures and object parts learnt from 1.2 million ImageNet photos, and for these classes they are almost linearly separable. The dense head only has to draw the boundaries between the classes in this feature space, which needs few parameters and therefore few training images. Augmentation adds nothing here (both models reach 0.925), because most of the work is done by the pretrained features.
2. Precomputed features assume that every image always gives the same feature vector, so the base runs only once. Augmentation changes the images in every epoch, so the base has to run on every image in every epoch. Precomputed features gain speed (seconds instead of minutes); augmentation gains robustness to position, orientation and size, at the cost of computing time.
3. A CNN trained from scratch on fewer than 60 images per class would overfit quickly and reach a low validation accuracy. Its filters start random, so it would have to learn edges, textures and shapes from these few images, which a pretrained base already provides.

## References

- Fei-Fei, L., Fergus, R. and Perona, P. (2004). Learning generative visual models from few training examples. [Caltech-101 dataset](https://data.caltech.edu/records/mzrjq-6wc02), CC BY 4.0.
- Keras guide: [Transfer learning and fine-tuning](https://keras.io/guides/transfer_learning/).